# Same operation on a lot of data, or different operations on different data?

No GPU is needed to see the principle: NumPy applies one operation to a whole
array at once, in the SIMD style, while a Python loop decides element by
element.

In [1]:
import numpy as np, time
x = np.random.default_rng(0).normal(size=1_000_000)

t0 = time.perf_counter(); y = [v * 2.0 + 1.0 for v in x];  t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); y = x * 2.0 + 1.0;                t_vec  = time.perf_counter() - t0
print(f"same arithmetic on 1,000,000 numbers:  loop {t_loop:.3f} s   vectorized {t_vec:.4f} s   ({t_loop / t_vec:.0f}x)")

same arithmetic on 1,000,000 numbers:  loop 0.096 s   vectorized 0.0072 s   (13x)


## A branch inside the loop

Vectorized code cannot "take a branch" per element; it computes both sides and
selects. That is what a GPU does too, and it is why divergent branches waste
lanes.

In [2]:
t0 = time.perf_counter(); y = [v * 2 if v > 0 else -v for v in x];          t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); y = np.where(x > 0, x * 2, -x);                    t_vec  = time.perf_counter() - t0
print(f"with a branch:  loop {t_loop:.3f} s   vectorized {t_vec:.4f} s   ({t_loop / t_vec:.0f}x): both sides computed for every element")

with a branch:  loop 0.081 s   vectorized 0.0147 s   (6x): both sides computed for every element


## When the work is genuinely different per element

An agent-based step where each agent runs its own small program does not
vectorize, and that is the shape of code that runs *slower* on a GPU than on a
CPU core.

In [3]:
def agent_step(state):
    # different agents do different things: no single operation to broadcast
    if state["mood"] > 0.5:
        state["friends"] += 1
    elif state["friends"] > 3:
        state["mood"] += 0.1
    else:
        state["mood"] -= 0.05
    return state

agents = [{"mood": m, "friends": f} for m, f in zip(np.random.default_rng(1).random(200_000), np.random.default_rng(2).integers(0, 6, 200_000))]
t0 = time.perf_counter(); agents = [agent_step(a) for a in agents]; print(f"200,000 branchy agent steps: {time.perf_counter() - t0:.3f} s: CPU work")

200,000 branchy agent steps: 0.030 s: CPU work


**What we used:** Flynn's taxonomy, SIMD versus MIMD, and why branches hurt
throughput (session 12, sections 6 to 8). Vectorize before buying hardware.